# NB03 – Daten Analyse

**Grid-Arbitrage** · Batteriespeicher-Arbitrage im Schweizer Strommarkt

**Gruppe:** SC26_Gruppe_2 | **Verantwortlich:** Senthuran Elankeswaran | **Datum:** März 2026

---

*Preisstruktur, [Dispatch](../organisation/O_02_Glossar.ipynb#g-dispatch)-Simulation, Wirtschaftlichkeit, [Netzentlastung](../organisation/O_02_Glossar.ipynb#g-netzentlastung), Spread-Zeitreihe.*


| [← NB02 Daten Bereinigung](02_Daten_Bereinigung.ipynb) | [↑ Übersicht ↑](../organisation/O_01_Project_Overview.ipynb) | [NB04 Visualisierungen →](04_Visualisierungen.ipynb) |
|:---|:---:|---:|

## Inhaltsverzeichnis<a id='toc_NB_03'></a>

[Einleitung](#einleitung_NB_03)  
[Initialisierung](#initialisierung_NB_03)  
1 [Preisstruktur-Analyse](#preisstruktur-analyse_NB_03)  
2 [Batterie-Dispatch-Simulation](#batterie-dispatch-simulation_NB_03)  
3 [Wirtschaftlichkeitsrechnung](#wirtschaftlichkeitsrechnung_NB_03)  
4 [Netzentlastungsszenarien](#netzentlastungsszenarien_NB_03)  
5 [Spread- & Volatilitäts-Zeitreihe](#spread-volatilitaets-zeitreihe_NB_03)  
[Fazit](#fazit_NB_03)  
[Abschluss](#abschluss_NB_03)  


---
## Einleitung <a id='einleitung_NB_03'></a>

[↑ Inhaltsverzeichnis](#toc_NB_03)

Analyse der bereinigten Spot-Preise aus NB02:

1. **Preisstruktur** — Tages- und Saisonprofile, Spread-Verteilungen
2. **Dispatch-Simulation** — Quantil-basierter Lade/Entlade-Algorithmus
   (P25 laden, P75 entladen) mit konfigurierbarem SoC-Band und Wirkungsgrad
3. **Wirtschaftlichkeit** — CAPEX, ROI, Payback je Segment
   (Privat 10 kWh / Gewerbe 100 kWh / Industrie 1 MWh / Utility 10 MWh)
4. **Netzentlastungsszenarien** — Rollout-Effekt bei verschiedenen Gleichzeitigkeitsraten
5. **Spread- und Volatilitätsverlauf** — monatlich aggregiert

Alle Parameter aus `../sync/config.json` (SSOT). Ergebnisse nach
`../sync/transfer.json` für NB04 und die Kür-Notebooks.


## Initialisierung<a id='initialisierung_NB_03'></a>

[↑ Inhaltsverzeichnis](#toc_NB_03)

Lädt `../sync/config.json` (SSOT), setzt Verzeichnispfade, Simulation- und Wirtschaftlichkeitsparameter sowie Datenzeitraum und Simulationsergebnisse aus `../sync/transfer.json`.

**Setup – Konfiguration & Verzeichnisstruktur:** Lädt `../sync/config.json`, setzt Pfade
und liest `FORCE_RELOAD`, `LIFETIME` und `ZIEL_ROI` als lokale Aliases.


In [ ]:
# ── lib/ aus Projekt-Root erreichbar machen + lib-Imports ───────────────────
# Notebook liegt in einem Unterordner (kuer/, experimental/, notebooks/,
# organisation/). Damit 'from lib.xxx import ...' funktioniert, muss der
# Projekt-Root vorne in sys.path stehen. autoreload sorgt dafür, dass
# Änderungen in lib/*.py ohne Kernel-Restart übernommen werden.
import sys, os
_PROJECT_ROOT = os.path.abspath('..')
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass

# lib-Imports (einmal zentral — in allen folgenden Zellen verfügbar)
from lib.plotting import show_source
from lib.io_ops   import load_transfer, save_transfer, log_dataindex, needs_rebuild, log_missing, final_check
from lib.simulation  import simulate_battery_dispatch

print(f'lib-Pfad aktiv: {_PROJECT_ROOT}/lib')


In [ ]:
import os, sys, warnings, json
import numpy  as np
import pandas as pd
from datetime import datetime
warnings.filterwarnings('ignore')

# Versionen anzeigen für Reproduzierbarkeit
print(f"Numpy        Version: {np.__version__}")
print(f"Pandas       Version: {pd.__version__}")
print(f"\U0001f4c5 Zuletzt ausgeführt am: {datetime.now().strftime('%d.%m.%Y um %H:%M:%S')}")

**🔎 Quellcode der importierten lib-Funktion**

Die Funktion `load_transfer` wird aus `lib/io_ops.py` importiert und
liest Einträge aus `sync/transfer.json`. Aufklappbar ist der Quellcode einsehbar.


In [ ]:
show_source(load_transfer)


In [ ]:
# ── ../sync/config.json laden (Single Source of Truth) ───────────────────────────────
# NEVER hardcode MODE oder FORCE_RELOAD hier — immer aus ../sync/config.json lesen
with open('../sync/config.json') as _f:
    CFG = json.load(_f)

MODE         = CFG['mode']
FORCE_RELOAD = CFG['force_reload']

# ── Sim-Parameter aus ../sync/config.json ────────────────────────────────────────────
_sim         = CFG['pflicht']['simulation']
CHARGE_Q     = _sim['charge_quantile']      # 0.25
DISCHARGE_Q  = _sim['discharge_quantile']   # 0.75
SOC_MIN_PCT  = _sim['soc_min_pct']          # 0.05
SOC_MAX_PCT  = _sim['soc_max_pct']          # 0.95
EFFICIENCY   = _sim['efficiency_roundtrip'] # 0.92

# ── Wirtschaftlichkeits-Parameter ─────────────────────────────────────────────
_w           = CFG['pflicht']['wirtschaftlichkeit']
CAPEX_EUR_KWH = _w['capex_eur_kwh']
OPEX_RATE    = _w['opex_rate']
LIFETIME     = _w['lifetime_j']
ZIEL_ROI_PCT = round(100 / LIFETIME, 2)  # Ziel-ROI = 1/lifetime_j: ROI der nötig ist, um BE in LIFETIME Jahren zu erreichen

# ── Szenario-Parameter (Gleichzeitigkeit) ─────────────────────────────────────
_sz           = CFG['szenarien']
GLEICHZEITIGKEIT = _sz['gleichzeitigkeit_aktiv']          # 'pessimistisch'|'realistisch'|'optimistisch'
RATE          = _sz['optionen'][GLEICHZEITIGKEIT]['rate']
SZ_OPT        = _sz['optionen'][GLEICHZEITIGKEIT]         # alle Parameter des aktiven Szenarios
CH_SPITZENLAST_GW = _sz['ch_spitzenlast_gw']


In [ ]:
DIR_RAW          = os.path.join('../data', 'raw')
DIR_PROCESSED    = os.path.join('../data', 'processed')
# intermediate: Szenario-abhängige Dateien in Unterordner
DIR_INTER_BASE   = os.path.join('../data', 'intermediate')
DIR_INTER_SZ     = os.path.join(DIR_INTER_BASE, GLEICHZEITIGKEIT)   # z.B. data/intermediate/realistisch/
DIR_INTER        = DIR_INTER_BASE  # szenario-unabhängige Dateien (spread, import/export)
DATAINDEX        = '../sync/dataindex.csv'

for d in [DIR_RAW, DIR_PROCESSED, DIR_INTER_BASE, DIR_INTER_SZ]:
    os.makedirs(d, exist_ok=True)

print(f'../sync/config.json  : geladen')
print(f'MODE         : {MODE}')
print(f'Gleichz.     : {GLEICHZEITIGKEIT} ({RATE*100:.0f}%)')
print(f'raw          : {os.path.abspath(DIR_RAW)}')
print(f'processed    : {os.path.abspath(DIR_PROCESSED)}')
print(f'intermediate : {os.path.abspath(DIR_INTER_SZ)} (szenario-abhängig)')


In [ ]:
# -- Transfer: Ergebnisse aus ../sync/transfer.json laden ----------------------------
TF       = load_transfer()
_dt      = TF.get('datenzeitraum', {})
_sim_tf  = TF.get('simulation', {})
TF_N_YEARS = _dt.get('n_years', None)
TF_START   = _dt.get('start_date', 'unbekannt')
TF_END     = _dt.get('end_date',   'unbekannt')
TF_SPREAD  = _sim_tf.get('spread_mean_eur_mwh', None)
TF_ECON    = _sim_tf.get('wirtschaftlichkeit', {})
TF_HYB     = TF.get('hybrid_simulation', {}).get('ergebnisse', {})
TF_KUER    = CFG.get('kuer_aktiv', {})
if TF:
    print(f"../sync/transfer.json: {TF_START} – {TF_END} ({TF_N_YEARS} Jahre) | Spread: {TF_SPREAD} EUR/MWh")


**Transfer NB01 → NB02:** Datenzeitraum (`n_years`) aus `../sync/transfer.json` lesen —
SSOT für alle Jahresdurchschnitts-Berechnungen. Fehlt die Datei: Fallback auf Selbstberechnung.


In [ ]:
# -- Transfer: n_years von NB01 übernehmen (SSOT: gleiche Methode überall) ----
_dt_r  = load_transfer(key='datenzeitraum', default={})
n_years = _dt_r.get('n_years', None)
_sz_tf  = _dt_r.get('_szenario_aktiv', None)
if n_years:
    print(f'n_years aus ../sync/transfer.json (NB01): {n_years:.3f}')
if _sz_tf and _sz_tf != SZ_AKTIV:
    print(f'⚠️  Szenario-Konflikt: transfer={_sz_tf} vs config={SZ_AKTIV}')
    print('   NB01 mit aktuellem Szenario neu ausführen!')


**Hilfsfunktionen:** `log_dataindex()` für das Datenprotokoll (identisch zu NB01,
hier neu definiert damit NB02 eigenständig ausführbar bleibt).


**🔎 Quellcode der importierten lib-Funktion**

Die Funktion `log_dataindex` wird aus `lib/io_ops.py` importiert und
schreibt Einträge ins Daten-Provenienz-Protokoll `sync/dataindex.csv`.
Bereits bestehende Einträge für denselben Dateinamen werden als
`superseded` markiert. Aufklappbar darunter ist der Quellcode einsehbar.


In [ ]:
show_source(log_dataindex)


**Rohdaten laden:** Spot-Preise und Netzlast aus `data/processed/` (bereinigt, NB02-Output) und Netzlast aus `data/raw/` einlesen;
Zeitfeatures (`hour`, `month`, `season`) berechnen.


In [ ]:
# ── Daten laden (NB02-Output) ────────────────────────────────────────────────
# Spot-Preise: aus data/processed/ (bereinigt von NB02), nicht raw!
# Netzlast:    aus data/raw/ (keine Bereinigung nötig — NB01-Output)
df_prices = pd.read_csv(os.path.join(DIR_PROCESSED, 'ch_spot_prices_clean.csv'))
df_prices['timestamp'] = pd.to_datetime(df_prices['timestamp'], utc=True)

# Zeitfeatures (werden HIER berechnet — NB02 speichert nur bereinigte Preise)
if 'hour' not in df_prices.columns:
    df_prices['hour']  = df_prices['timestamp'].dt.hour
if 'month' not in df_prices.columns:
    df_prices['month'] = df_prices['timestamp'].dt.month
if 'season' not in df_prices.columns:
    _season_map = {12:'Winter', 1:'Winter',  2:'Winter',
                    3:'Frühling',4:'Frühling',5:'Frühling',
                    6:'Sommer',  7:'Sommer',  8:'Sommer',
                    9:'Herbst', 10:'Herbst', 11:'Herbst'}
    df_prices['season'] = df_prices['month'].map(_season_map)

df_load = pd.read_csv(os.path.join(DIR_RAW, 'ch_netzlast_raw.csv'))
df_load['timestamp'] = pd.to_datetime(df_load['timestamp'], utc=True)

print(f'MODE       : {MODE}')
print(f'Preisdaten : {len(df_prices):,} Zeilen | {list(df_prices.columns)}')
print(f'Lastdaten  : {len(df_load):,} Zeilen  | {list(df_load.columns)}')


---
## 1. Preisstruktur-Analyse <a id='preisstruktur-analyse_NB_03'></a>

[↑ Inhaltsverzeichnis](#toc_NB_03)

Berechnet stündliche Durchschnittspreise und den Intra-Tag-Spread (p75 − p25).
Der Spread ist der direkte Erlöstreiber — je höher, desto rentabler die Arbitrage.


In [ ]:
# ── Tagesprofil & Spread ──────────────────────────────────────────────────────
h_stats   = df_prices.groupby('hour')['price_eur_mwh'].agg(mean='mean',std='std').round(2)
low_h     = h_stats['mean'].nsmallest(4).index.tolist()
hi_h      = h_stats['mean'].nlargest(4).index.tolist()
spread    = h_stats.loc[hi_h,'mean'].mean() - h_stats.loc[low_h,'mean'].mean()

print(f'Guenstigste Stunden  : {sorted(low_h)}  Ø {h_stats.loc[low_h,"mean"].mean():.1f} EUR/MWh')
print(f'Teuerste Stunden     : {sorted(hi_h)}   Ø {h_stats.loc[hi_h,"mean"].mean():.1f} EUR/MWh')
print(f'Arbitrage-Spread (Ø) : {spread:.1f} EUR/MWh')
sn = {0:'Winter',1:'Frühling',2:'Sommer',3:'Herbst'}
for s, avg in df_prices.groupby('season')['price_eur_mwh'].mean().items():
    print(f'  {sn[s]:<10}: {avg:.1f} EUR/MWh')


---
## 2. Batterie-Dispatch-Simulation <a id='batterie-dispatch-simulation_NB_03'></a>

[↑ Inhaltsverzeichnis](#toc_NB_03)

Tagesbasierte Schwellenwert-Strategie: Laden bei Preis ≤ [p25](../organisation/O_02_Glossar.ipynb#g-p25p75), Einspeisen bei Preis ≥ [p75](../organisation/O_02_Glossar.ipynb#g-p25p75).
Zuerst die [Dispatch](../organisation/O_02_Glossar.ipynb#g-dispatch)-Funktion definieren, dann alle vier Segmente über den
gesamten Datensatz simulieren und den Jahresdurchschnitt bilden.


**Dispatch-Simulation:** Alle vier Segmente (Privat / Gewerbe / Industrie / Utility)
über den gesamten Datensatz simulieren; Jahresdurchschnitt aus `n_years` bilden.


**🔎 Quellcode der importierten lib-Funktion**

Die Funktion `simulate_battery_dispatch` wird aus `lib/simulation.py` importiert und
implementiert das Schwellenwert-Dispatch-Modell auf Basis tages-
basierter Preisquantile. Die Kern-Logik wurde aus NB03 extrahiert
und konsolidiert NB03.simulate_battery und K_06.simulate_battery_reactive.
Aufklappbar ist der Quellcode einsehbar.


In [ ]:
show_source(simulate_battery_dispatch)


In [ ]:
import json  # Re-Import mit lokalem Alias (Simulations-Zelle)

# ── Simulation aller Segmente ─────────────────────────────────────────────────
# Simuliert alle verfügbaren Daten und bildet den Jahresdurchschnitt (sum / n_years).
# Optimierte Dispatch-Funktion: O(n) statt O(n²), tqdm Progress-Bar pro Segment.
# Typische Laufzeit: ~5-15 Sekunden für alle 4 Segmente.

SEGMENTS = [('Privat_10kWh',10,5,200_000),('Gewerbe_100kWh',100,30,10_000),
            ('Industrie_1MWh',1_000,200,1_000),('Utility_10MWh',10_000,1_000,100)]

# Alle verfügbaren Preisdaten (kein df_sample mehr)
df_sim = df_prices.copy()
# n_years aus ../sync/transfer.json (NB01 SSOT); Fallback: selbst berechnen
if n_years is None:
    n_years = df_sim['timestamp'].dt.year.nunique()
    print(f'n_years Fallback (selbst berechnet): {n_years}')
# n_years jetzt verfügbar:
results = {}

print(f'Simulationszeitraum: {df_sim["timestamp"].min().date()} – '
      f'{df_sim["timestamp"].max().date()} ({n_years} Jahre)')
print(f'{"Segment":<22} {"Jahreserloes":>13} {"EUR/kWh/J":>11} {"Lade-h/J":>10} {"Einsp-h/J":>10}')
print('-'*70)

for name, cap, pwr, units in SEGMENTS:
    res         = simulate_battery_dispatch(
                      df_sim, cap, pwr,
                      efficiency=EFFICIENCY,
                      charge_q=CHARGE_Q,
                      discharge_q=DISCHARGE_Q,
                      soc_min_pct=SOC_MIN_PCT,
                      soc_max_pct=SOC_MAX_PCT)
    annual_rev  = res['cashflow_eur'].sum() / n_years   # Ø über alle Jahre
    rev_per_kwh = annual_rev / cap
    n_ch = (res['action']=='charge').sum()    / n_years
    n_di = (res['action']=='discharge').sum() / n_years
    results[name] = dict(cap=cap, pwr=pwr, units=units, annual_rev=annual_rev,
                          rev_per_kwh=rev_per_kwh, n_charge=n_ch, n_discharge=n_di)
    print(f'{name:<22} {annual_rev:>12.0f}€ {rev_per_kwh:>10.2f} '
          f'{n_ch:>10.0f} {n_di:>10.0f}')

print('\nSimulation abgeschlossen (Ø Jahreserlös über gesamten Datenzeitraum).')


---
## 3. Wirtschaftlichkeitsrechnung <a id='wirtschaftlichkeitsrechnung_NB_03'></a>

[↑ Inhaltsverzeichnis](#toc_NB_03)

Berechnet [CAPEX](../organisation/O_02_Glossar.ipynb#g-capex), jährliche Erlöse, [ROI](../organisation/O_02_Glossar.ipynb#g-roi) und Amortisationsdauer für alle Segmente.
Ergebnis: `wirtschaftlichkeit.csv` und `netzentlastung_szenarien.csv` in `intermediate/<szenario>/`.


In [ ]:
# ── CAPEX / ROI / Amortisation → intermediate/ ────────────────────────────────
# CAPEX_EUR_KWH, OPEX_RATE, LIFETIME bereits aus ../sync/config.json geladen (Setup-Zelle)

econ = {}
print(f'{"Segment":<22} {"CAPEX":>9} {"Jährl.Erlös":>12} {"OPEX":>8} {"Netto":>8} {"Amort.":>9} {"ROI":>7}')
print('-'*83)

for name, res in results.items():
    cap, annual_rev = res['cap'], res['annual_rev']
    capex = CAPEX_EUR_KWH[name]*cap
    opex  = capex*OPEX_RATE
    net   = annual_rev-opex
    pb    = capex/net if net>0 else float('inf')
    roi   = net/capex*100
    econ[name] = dict(segment=name, capex=capex, annual_rev=annual_rev,
                       opex_annual=opex, net_annual=net,
                       payback_years=pb, roi_pct=roi, rev_per_kwh=res['rev_per_kwh'])
    pb_s = f'{pb:.1f}J' if pb<50 else '>50J'
    print(f'{name:<22} {capex:>8.0f}€ {annual_rev:>11.0f}€ {opex:>7.0f}€ '
          f'{net:>7.0f}€ {pb_s:>9} {roi:>6.1f}%')

df_econ = pd.DataFrame(list(econ.values()))
ECON_FILE = os.path.join(DIR_INTER_SZ, 'wirtschaftlichkeit.csv')  # szenario-abhängig
df_econ.to_csv(ECON_FILE, index=False)
log_dataindex('wirtschaftlichkeit.csv','NB2:dispatch', ECON_FILE,'intermediate',
              rows=len(df_econ), size_kb=os.path.getsize(ECON_FILE)/1024, note=f'MODE={MODE}')
print(f'\nGespeichert: {ECON_FILE}')
print('Hinweis: Nur Arbitrage-Erloese – Regelenergie (FCR/aFRR) nicht eingerechnet.')


**Verifikation:** [CAPEX](../organisation/O_02_Glossar.ipynb#g-capex), [ROI](../organisation/O_02_Glossar.ipynb#g-roi) und Amortisation der vier Segmente tabellarisch prüfen.


In [ ]:
# ── Verifikation: Wirtschaftlichkeit ────────────────────────────────────────
print(f'Segmente: {len(df_econ)}')
df_econ[['segment','capex','net_annual','roi_pct','payback_years']].head()


---
## 4. Netzentlastungsszenarien <a id='netzentlastungsszenarien_NB_03'></a>

[↑ Inhaltsverzeichnis](#toc_NB_03)

### Modell: Aggregierter Rollout-Effekt

Die vier Szenarien beschreiben mögliche Rollout-Zustände des CH-Batteriemarkts.
Die Zahlen sind **Schätzungen** basierend auf Markttrends und [BFE](../organisation/O_02_Glossar.ipynb#g-bfe)-Prognosen —
keine offiziellen Zielvorgaben.

| Szenario | Jahr | Privat | Gewerbe | Industrie |
|---|---|---|---|---|
| Status Quo | 2024 | 0 | 0 | 0 |
| Moderat | 2027 | 50'000 | 2'000 | 200 |
| Ambitioniert | 2030 | 200'000 | 8'000 | 800 |
| Transformativ | 2035 | 800'000 | 30'000 | 2'000 |

**Berechnung [Netzentlastung](../organisation/O_02_Glossar.ipynb#g-netzentlastung):**
```
Entlastung [MW] = (n_privat×5kW + n_gewerbe×30kW + n_industrie×200kW) / 1000 × Rate
```

**Gleichzeitigkeits-Schalter** (`GLEICHZEITIGKEIT` in der Code-Zelle):

| Modus | Rate | Annahme |
|---|---|---|
| `'optimistisch'` | 70%⚙ | Koordinierter [VPP](../organisation/O_02_Glossar.ipynb#g-vpp)-[Dispatch](../organisation/O_02_Glossar.ipynb#g-dispatch) — alle Batterien reagieren aufs gleiche Signal |
| `'realistisch'` | 40%⚙ | Unkoordinierter Markt — jede Batterie dispatcht eigenständig |

> **Default:** `'realistisch'` — die Berichte (NB4) referenzieren den gesetzten Wert
> automatisch. Um die optimistische Variante zu berechnen: Schalter umstellen und
> NB2 + NB3 neu ausführen.


In [ ]:
# ── Szenarien → intermediate/<szenario>/ ──────────────────────────────────────
# GLEICHZEITIGKEIT und RATE kommen aus ../sync/config.json (bereits in Setup geladen)
# Szenario-Outputs in DIR_INTER_SZ = data/intermediate/<gleichzeitigkeit>/

KW = {'Privat_10kWh': 5, 'Gewerbe_100kWh': 30, 'Industrie_1MWh': 200}

SCENARIOS = [
    ('Status Quo (2024)',    0,
     SZ_OPT.get('n_privat_2027',0)*0,    # 0 für Status Quo
     SZ_OPT.get('n_gewerbe_2027',0)*0,
     SZ_OPT.get('n_industrie_2027',0)*0),
    ('Moderat (2027)',
     SZ_OPT.get('n_privat_2027', 50_000),
     SZ_OPT.get('n_gewerbe_2027', 2_000),
     SZ_OPT.get('n_industrie_2027', 200)),
    ('Ambitioniert (2030)',
     SZ_OPT.get('n_privat_2030', 200_000),
     SZ_OPT.get('n_gewerbe_2030', 8_000),
     SZ_OPT.get('n_industrie_2030', 800)),
    ('Transformativ (2035)',
     SZ_OPT.get('n_privat_2035', 800_000),
     SZ_OPT.get('n_gewerbe_2035', 30_000),
     SZ_OPT.get('n_industrie_2035', 2_000)),
]

print(f'Gleichzeitigkeit : {GLEICHZEITIGKEIT} ({RATE*100:.0f}%)')
print(f'CH Systemspitze  : {CH_SPITZENLAST_GW} GW')
print(f'Szenario-Output  : {DIR_INTER_SZ}')
print()
print(f'{"Szenario":<28} {"Entlastung [MW]":>16} {"Reduktion":>10}')
print('-'*58)

rows = []
for sc_data in SCENARIOS:
    if len(sc_data) == 4:
        sc, n_p, n_g, n_i = sc_data
    else:
        sc = sc_data[0]; n_p = sc_data[1]; n_g = sc_data[2]; n_i = sc_data[3]
    mw = (n_p*KW['Privat_10kWh'] + n_g*KW['Gewerbe_100kWh'] +
          n_i*KW['Industrie_1MWh']) / 1000 * RATE
    rows.append({
        'szenario':            sc,
        'gleichzeitigkeit':    GLEICHZEITIGKEIT,
        'rate_pct':            RATE * 100,
        'n_privat':            n_p,
        'n_gewerbe':           n_g,
        'n_industrie':         n_i,
        'entlastung_mw':       round(mw, 1),
        'neue_spitzenlast_gw': round(max(0, CH_SPITZENLAST_GW - mw/1000), 3),
        'reduktion_pct':       round(mw / 1000 / CH_SPITZENLAST_GW * 100, 2),
    })
    print(f'{sc:<28} {mw:>15.0f} MW  {mw/1000/CH_SPITZENLAST_GW*100:>9.1f}%')

df_sz = pd.DataFrame(rows)
SZ_FILE = os.path.join(DIR_INTER_SZ, 'netzentlastung_szenarien.csv')
df_sz.to_csv(SZ_FILE, index=False)
log_dataindex('netzentlastung_szenarien.csv', 'NB02:szenario', SZ_FILE,
              'intermediate',  # Szenario steht in note= (data_type bleibt generisch)
              rows=len(df_sz),
              size_kb=os.path.getsize(SZ_FILE)/1024,
              note=f'MODE={MODE}, Gleichzeitigkeit={GLEICHZEITIGKEIT} ({RATE*100:.0f}%)')
print(f'\nGespeichert: {SZ_FILE}')


---
## 5. Spread- & Volatilitäts-Zeitreihe <a id='spread-volatilitaets-zeitreihe_NB_03'></a>

[↑ Inhaltsverzeichnis](#toc_NB_03)

Berechnet monatlich: Intra-Tag-Spread ([p75−p25](../organisation/O_02_Glossar.ipynb#g-p25p75) Median), [Volatilität](../organisation/O_02_Glossar.ipynb#g-volatilitaet) und
Negativpreis-Stunden → `spread_zeitreihe.csv` in `intermediate/`.

| Kennzahl | Berechnung | Bedeutung |
|---|---|---|
| `spread_median` | Median(p75−p25 pro Tag) | Typisches tägliches Arbitrage-Fenster |
| `volatility` | Std. Abweichung Stundenpreise | Allgemeine Marktvolatilität |
| `neg_price_h` | Stunden mit Preis < 0 | Solar-Überschuss-Indikator |

> Chart 7 (NB3) visualisiert diese Zeitreihe mit Trendlinie.
> Jährlich neu ausführen um Trend zu aktualisieren.


**🔎 Quellcode der importierten lib-Funktion**

`needs_rebuild` aus `lib.io_ops` — aufklappbar ist der Quellcode einsehbar.


In [ ]:
show_source(needs_rebuild)


In [ ]:
# ── 6. Spread/Volatilität Zeitreihe → intermediate/ ─────────────────────────
SPREAD_FILE = os.path.join(DIR_INTER, 'spread_zeitreihe.csv')

if not needs_rebuild(SPREAD_FILE, 12, 'spread_ts'):
    print(f'spread_zeitreihe.csv vorhanden ({os.path.getsize(SPREAD_FILE)/1024:.0f} KB) – kein Rebuild.')
    df_spread = pd.read_csv(SPREAD_FILE, parse_dates=['yearmonth'])
else:
    print('Berechne Spread-Zeitreihe...')
    CLEAN_FILE = os.path.join(DIR_PROCESSED, 'ch_spot_prices_clean.csv')
    df_clean = pd.read_csv(CLEAN_FILE, parse_dates=['timestamp'])
    df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'], utc=True)
    df_clean['date']      = df_clean['timestamp'].dt.date
    df_clean['yearmonth'] = df_clean['timestamp'].dt.to_period('M').dt.to_timestamp()

    # Intra-Tag Spread pro Tag
    day_stats = df_clean.groupby('date')['price_eur_mwh'].agg(
        p25=lambda x: x.quantile(0.25),
        p75=lambda x: x.quantile(0.75),
    ).reset_index()
    day_stats['spread']    = day_stats['p75'] - day_stats['p25']
    day_stats['yearmonth'] = pd.to_datetime(day_stats['date']).dt.to_period('M').dt.to_timestamp()

    # Monatliche Aggregation
    spread_m = day_stats.groupby('yearmonth')['spread'].median().reset_index()
    spread_m.columns = ['yearmonth', 'spread_median']

    vol_m = df_clean.groupby('yearmonth').agg(
        volatility  = ('price_eur_mwh', 'std'),
        mean_price  = ('price_eur_mwh', 'mean'),
        neg_price_h = ('price_eur_mwh', lambda x: (x < 0).sum()),
    ).reset_index()

    df_spread = spread_m.merge(vol_m, on='yearmonth')
    df_spread.to_csv(SPREAD_FILE, index=False)
    log_dataindex('spread_zeitreihe.csv', 'NB2:spread_analyse', SPREAD_FILE,
                  'intermediate', rows=len(df_spread),
                  size_kb=os.path.getsize(SPREAD_FILE)/1024,
                  note='Monatlicher Intra-Tag-Spread, Volatilität, Negativpreise')
    print(f'Gespeichert: {SPREAD_FILE} | {len(df_spread)} Monate')
    print(f'  Zeitraum: {df_spread["yearmonth"].min().strftime("%Y-%m")} – '
          f'{df_spread["yearmonth"].max().strftime("%Y-%m")}')
    print(f'  Ø Spread: {df_spread["spread_median"].mean():.1f} EUR/MWh '
          f'| Min: {df_spread["spread_median"].min():.1f} '
          f'| Max: {df_spread["spread_median"].max():.1f}')
    print(f'  Negativpreis-Stunden total: {df_spread["neg_price_h"].sum():,}')


**Verifikation:** Spread-Zeitreihe (monatlich) auf Shape und Wertebereich prüfen.


In [ ]:
# ── Verifikation: Spread-Zeitreihe ───────────────────────────────────────────
print(f'Shape  : {df_spread.shape}')
print(f'Spalten: {list(df_spread.columns)}')
df_spread.head(3)


**Transfer NB02 → downstream:** Simulation-Kennzahlen und Datenzeitraum in
`../sync/transfer.json` schreiben — wird von NB04, K_00, K_05, K_10, K_99 gelesen.


**🔎 Quellcode der importierten lib-Funktion**

Die Funktion `save_transfer` wird aus `lib/io_ops.py` importiert und
in der folgenden Zelle verwendet. Aufklappbar ist der Quellcode einsehbar.


In [ ]:
show_source(save_transfer)


In [ ]:
# ── Saisonal-Kennzahlen für transfer.json ────────────────────────────────────
# Liefert je Saison:
#   spread_max_min_eur_mwh : Mittlere TAGES-MAX-MIN-Spanne (theoretische Obergrenze
#                            der Arbitrage; täglicher Höchstpreis − Tiefstpreis,
#                            Saison-Mittel)
#   monatserloese          : p75−p25-Spread der Stundenmittel pro Monat
#                            (Basis der simulierten Erlöse, vgl. Chart 5b in NB04)
SAISON_MONATE = {'Winter': [12, 1, 2], 'Frühling': [3, 4, 5],
                 'Sommer': [6, 7, 8],  'Herbst':   [9, 10, 11]}
MONAT_DE = {1:'Jan', 2:'Feb',  3:'Mär',  4:'Apr',
            5:'Mai', 6:'Jun',  7:'Jul',  8:'Aug',
            9:'Sep', 10:'Okt', 11:'Nov', 12:'Dez'}

# Tagesweise max-min-Spanne berechnen (echte Marktpreisspanne pro Tag)
_df_p = df_prices.copy()
_df_p['date'] = _df_p['timestamp'].dt.date
_daily = _df_p.groupby('date').agg(
    daily_max = ('price_eur_mwh', 'max'),
    daily_min = ('price_eur_mwh', 'min'),
    month     = ('month',         'first'),
).reset_index()
_daily['daily_span'] = _daily['daily_max'] - _daily['daily_min']

_saisonal = {}
for _s_name, _monate in SAISON_MONATE.items():
    # Theoretische Obergrenze: Mittel der täglichen max-min Spannen
    _s_daily = _daily[_daily['month'].isin(_monate)]
    _spread_max_min = float(_s_daily['daily_span'].mean())

    # Monatserlöse: p75−p25-Spread der Stundenmittel pro Monat
    _monatserloese = {}
    for _m in _monate:
        _df_m = df_prices[df_prices['month'] == _m]
        if len(_df_m) > 0:
            _hp = _df_m.groupby('hour')['price_eur_mwh'].mean()
            _sp = float(_hp.quantile(DISCHARGE_Q) - _hp.quantile(CHARGE_Q))
            _monatserloese[MONAT_DE[_m]] = round(_sp, 1)

    _saisonal[_s_name] = {
        'spread_max_min_eur_mwh': round(_spread_max_min, 1),
        'monatserloese':          _monatserloese,
    }

print('Saisonal-Kennzahlen berechnet:')
for _s, _v in _saisonal.items():
    print(f'  {_s:<10}: max-min={_v["spread_max_min_eur_mwh"]:>6.1f} EUR/MWh '
          f'| Monatserlöse: {_v["monatserloese"]}')


In [ ]:
# -- Transfer: Simulation-Ergebnisse in ../sync/transfer.json schreiben ---------------
# simulation-Dict lokal aufbauen, dann atomar schreiben
_sim_result = load_transfer(key='simulation', default={})

# Spread-Kennzahlen aus spread_zeitreihe.csv
if os.path.exists(SPREAD_FILE):
    _df_sp = pd.read_csv(SPREAD_FILE)
    _sim_result['spread_mean_eur_mwh']     = round(float(_df_sp['spread_median'].mean()), 2)
    _sim_result['spread_p25_mean']          = round(float(_df_sp['spread_median'].quantile(0.25)), 2)
    _sim_result['spread_p75_mean']          = round(float(_df_sp['spread_median'].quantile(0.75)), 2)
    # neg_price_h ist eine Spalte mit 1 Zeile pro MONAT → mean() wäre Monats-Mittel,
    # nicht Jahres-Wert. Korrekt: sum() (gesamte Negativpreis-Stunden) / n_years (Anzahl Jahre).
    _sim_result['neg_price_hours_per_year'] = round(float(_df_sp['neg_price_h'].sum()) / n_years, 1)

# n_years aus Preisdaten
_sim_result['n_years_sim'] = n_years

# Wirtschaftlichkeit je Segment (inkl. Lifetime-Aggregate)
_sim_result['wirtschaftlichkeit'] = {}
for _, row in df_econ.iterrows():
    _net_lifetime = float(row['net_annual']) * LIFETIME
    _sim_result['wirtschaftlichkeit'][row['segment']] = {
        'roi_pct':              round(float(row['roi_pct']), 2),
        'net_annual':           round(float(row['net_annual']), 1),
        'annual_rev':           round(float(row['annual_rev']), 1),
        'capex':                round(float(row['capex']), 0),
        'payback_years':        round(float(row['payback_years']), 1) if row['payback_years'] < 999 else 999,
        'rev_per_kwh_per_year': round(float(row['rev_per_kwh']), 2),
        'net_lifetime_keur':    round(_net_lifetime / 1000, 1),
        'rueckfluss_pct':       round(_net_lifetime / float(row['capex']) * 100, 1),
    }

# Netzentlastungs-Szenarien aus netzentlastung_szenarien.csv
if os.path.exists(SZ_FILE):
    _df_sz = pd.read_csv(SZ_FILE)
    _sim_result['netzentlastung'] = {
        row['szenario']: {
            'reduktion_pct': round(float(row['reduktion_pct']), 2),
            'entlastung_mw': round(float(row['entlastung_mw']), 1),
        }
        for _, row in _df_sz.iterrows()
    }

# Saisonal-Kennzahlen (vorher in eigener Cell als _saisonal berechnet)
_sim_result['saisonal'] = _saisonal

save_transfer(_sim_result, key='simulation')
print(f"../sync/transfer.json: simulation geschrieben")
print(f"  n_years={n_years:.2f} | spread_mean={_sim_result.get('spread_mean_eur_mwh','?')} EUR/MWh")
for seg, v in _sim_result['wirtschaftlichkeit'].items():
    print(f"  {seg}: ROI={v['roi_pct']}% | net={v['net_annual']:.0f} EUR/Jahr | "
          f"Erlös/kWh={v['rev_per_kwh_per_year']} | Netto{LIFETIME}J={v['net_lifetime_keur']} kEUR | "
          f"Rückfluss={v['rueckfluss_pct']}%")
for sc, v in _sim_result.get('netzentlastung', {}).items():
    print(f"  Szenario {sc:<24}: {v['reduktion_pct']}% Reduktion ({v['entlastung_mw']:.0f} MW)")
for s, v in _sim_result.get('saisonal', {}).items():
    print(f"  Saison {s:<10}: max-min={v['spread_max_min_eur_mwh']} EUR/MWh")


**Abschlusskontrolle:** Pflicht-Ausgabedateien validieren; dataindex-Status ausgeben.
Fehler müssen vor NB03 behoben werden.


---
## Fazit <a id='fazit_NB_03'></a>

[↑ Inhaltsverzeichnis](#toc_NB_03)

Wirtschaftlichkeit der Pflicht-Segmente im realistischen Szenario:

- **Kein Segment** erreicht den Break-Even-ROI (1/Lifetime = 8.33%) bei reiner Arbitrage:
  Privat 1.78%, Gewerbe 3.13%, Industrie 4.56%, Utility 2.95%
- Industrie kommt mit ~21.9 Jahren Payback dem Break-Even am nächsten,
  Privat mit ~56 Jahren am weitesten entfernt

Saisonalität (wichtig für strategische Empfehlungen):

- **Frühling** — höchster Spread (~84 EUR/MWh max–min Tagesspanne)
- **Sommer** — Solar-Mittagstief erzeugt phasenweise Negativpreise
- **Winter** — niedrigster Spread (~49 EUR/MWh)

Die vollständige Charts-Darstellung folgt in NB04; Transfer-Kennzahlen
sind in `../sync/transfer.json` für die Kür-Notebooks verfügbar.


---
## Abschluss<a id='abschluss_NB_03'></a>

[↑ Inhaltsverzeichnis](#toc_NB_03)

Pflicht-Ausgabedateien validieren; Übergabe-Kennzahlen in `../sync/transfer.json` prüfen.

**🔎 Quellcode der importierten lib-Funktion**

Die Funktion `final_check` wird aus `lib/io_ops.py` importiert und
in der folgenden Zelle verwendet. Aufklappbar ist der Quellcode einsehbar.

In [ ]:
show_source(final_check)

In [ ]:
# ── Abschlusskontrolle NB03 ──────────────────────────────────────────────────
final_check(
    'NB03',
    files=[
        (os.path.join(DIR_PROCESSED, 'ch_spot_prices_clean.csv'),
         'ch_spot_prices_clean.csv',                                  50_000),
        (os.path.join(DIR_INTER_SZ,  'wirtschaftlichkeit.csv'),
         f'wirtschaftlichkeit.csv [{GLEICHZEITIGKEIT}]',                 100),
        (os.path.join(DIR_INTER_SZ,  'netzentlastung_szenarien.csv'),
         f'netzentlastung_szenarien.csv [{GLEICHZEITIGKEIT}]',           100),
        (os.path.join(DIR_INTER,     'spread_zeitreihe.csv'),
         'spread_zeitreihe.csv',                                         500),
    ],
    weiter_msg='NB04 Visualisierungen',
    fehler_msg='NB04',
    extras=['Kür-Abhängigkeiten:',
            '  ch_import_export_analyse.csv → wird in K_02 berechnet (benötigt ch_crossborder_raw.csv)'],
)

| [← NB02 Daten Bereinigung](02_Daten_Bereinigung.ipynb) | [↑ Übersicht](../organisation/O_01_Project_Overview.ipynb) | [NB04 Visualisierungen →](04_Visualisierungen.ipynb) |
|:---|:---:|---:|